# 08 · Cointegración + Spread Analysis — Finance Concept

**Contexto:** En Pairs Trading (Stat Arb), dos activos individualmente no estacionarios pueden tener una combinación lineal estacionaria — están **cointegrados**. El spread entre ellos revierte a la media, lo que permite estrategias de trading market-neutral. Engle & Granger (Nobel 2003) formalizaron este concepto.

**Campo de origen:** Econometría (Engle & Granger 1987) → Stat Arbitrage → Pairs Trading  
**Dataset:** BVL — Alicorp (ALICORC1) + Backus (BACKUSI1) · pares de consumo masivo peruano

---

## Marco teórico

### Definición de cointegración

Dos series $X_t$ e $Y_t$ son **I(1)** (integradas de orden 1) si sus primeras diferencias son estacionarias. Son **cointegradas** si existe $\beta$ tal que:

$$Z_t = Y_t - \beta X_t \sim I(0) \quad \text{(estacionario)}$$

$Z_t$ es el **spread** — revierte a su media de largo plazo.

### Test de Engle-Granger (2 pasos)

**Paso 1:** Regresión de cointegración: $Y_t = \alpha + \beta X_t + \epsilon_t$  
**Paso 2:** Test ADF sobre $\hat{\epsilon}_t$ — si rechaza raíz unitaria → cointegrados

### Señal de trading del spread

$$z_t = \frac{Z_t - \mu_Z}{\sigma_Z} \qquad \text{señal} = \begin{cases} \text{Long spread} & z_t < -2 \\ \text{Short spread} & z_t > +2 \\ \text{Cerrar} & |z_t| < 0.5 \end{cases}$$

### Vector Error Correction Model (VECM)

$$\Delta Y_t = \alpha_Y (Y_{t-1} - \beta X_{t-1} - c) + \Gamma \Delta X_{t-1} + \epsilon_t$$

$\alpha_Y$ es la **velocidad de ajuste** — qué fracción del desequilibrio se corrige cada período.

**Referencias:** Engle, R.F. & Granger, C.W.J. (1987). *Econometrica* 55(2). Vidyamurthy, G. (2004). *Pairs Trading*. Wiley.

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
warnings.filterwarnings('ignore')
os.makedirs('data', exist_ok=True)

C = dict(
    ali='#1D4ED8',   bac='#15803D',   spread='#DC2626',
    fill_l='#DCFCE7', fill_s='#FEE2E2', neutral='#94A3B8',
    signal='#F59E0B', zero='#1E293B'
)
np.random.seed(42)
print('✓ OK')

In [ ]:
# ── DATOS — BVL: ALICORC1 + BACKUSI1 ─────────────────────────────────────────
# Fuente real:
#   import yfinance as yf
#   ali = yf.download('ALICORC1.LM', start='2015-01-01', auto_adjust=True)['Close']
#   bac = yf.download('BACKUSI1.LM', start='2015-01-01', auto_adjust=True)['Close']
#
# Estadísticos reales BVL 2015-2024:
#   ALICORC1: consumo masivo, correlación con commodities agrícolas
#   BACKUSI1: cerveza/bebidas, correlación con consumo interno
#   Correlación histórica: ~0.65-0.75 (ambas expuestas a consumo peruano)
#   Relación de cointegración: β ≈ 1.8-2.2 (Backus más caro por unidad)

try:
    import yfinance as yf
    ali_raw = yf.download('ALICORC1.LM', start='2015-01-01',
                           auto_adjust=True, progress=False)['Close'].dropna()
    bac_raw = yf.download('BACKUSI1.LM', start='2015-01-01',
                           auto_adjust=True, progress=False)['Close'].dropna()
    common = ali_raw.index.intersection(bac_raw.index)
    if len(common) > 200:
        ali = ali_raw.loc[common]
        bac = bac_raw.loc[common]
        SOURCE = 'Yahoo Finance — datos reales BVL'
        print(f'✓ {len(ali)} días descargados')
    else:
        raise ValueError('Datos insuficientes')
except Exception as e:
    SOURCE = 'Simulación calibrada (estadísticos reales ALICORC1 + BACKUSI1 BVL 2015-2024)'
    print(f'yfinance no disponible — {SOURCE}')

    n = 2200
    dates = pd.bdate_range('2015-01-05', periods=n)

    # Random walk cointegrado: Y_t = β·X_t + Z_t, Z_t ~ I(0)
    beta_true = 2.0
    # X = random walk (ALICORC1)
    rw_ali = np.cumsum(np.random.normal(0.0003, 0.009, n))
    ali    = pd.Series(10.0 * np.exp(rw_ali), index=dates, name='ALICORC1')

    # Spread estacionario con mean-reversion (AR(1) con φ=0.92)
    z = np.zeros(n)
    for t in range(1, n):
        z[t] = 0.92 * z[t-1] + np.random.normal(0, 0.8)

    # Backus = β·Alicorp + spread + drift
    bac_log = beta_true * rw_ali + 0.0002*np.arange(n) + z*0.02 + np.log(22)
    bac     = pd.Series(np.exp(bac_log), index=dates, name='BACKUSI1')

print(f'\nFuente : {SOURCE}')
print(f'Período: {ali.index[0].date()} → {ali.index[-1].date()}')
print(f'n      : {len(ali)} días hábiles')
print(f'ALICORC1: S/.{float(ali.mean()):.2f} media  σ={float(ali.std()):.2f}')
print(f'BACKUSI1: S/.{float(bac.mean()):.2f} media  σ={float(bac.std()):.2f}')

## Mini-EDA

In [ ]:
# ── EDA 1/2 — Tests de raíz unitaria ────────────────────────────────────────
def adf_summary(series, name):
    res = adfuller(series.dropna(), autolag='AIC')
    stat, pval = res[0], res[1]
    conclusion = 'No estacionaria (I(1))' if pval > 0.05 else 'Estacionaria (I(0))'
    print(f'  {name:<20} ADF={stat:7.3f}  p={pval:.4f}  → {conclusion}')
    return pval

print('── Test ADF — niveles (deben ser I(1)) ─────────────────────────')
p_ali = adf_summary(ali, 'ALICORC1')
p_bac = adf_summary(bac, 'BACKUSI1')

print('\n── Test ADF — primeras diferencias (deben ser I(0)) ────────────')
adf_summary(ali.diff().dropna(), 'ΔALICORC1')
adf_summary(bac.diff().dropna(), 'ΔBACKUSI1')

print('\n── Test de cointegración (Engle-Granger) ───────────────────────')
score, pval_coint, _ = coint(ali, bac)
print(f'  Estadístico: {score:.4f}  p-valor: {pval_coint:.4f}')
print(f'  → {"Cointegradas ✓" if pval_coint < 0.05 else "No cointegradas"}')

In [ ]:
# ── EDA 2/2 — Precios + scatter ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle(f'Mini-EDA — ALICORC1 + BACKUSI1 BVL · {SOURCE}', fontsize=10, y=1.01)

ax = axes[0]
ax2 = ax.twinx()
ax.plot(ali.index, ali.values,  color=C['ali'], lw=0.9, label='ALICORC1 (izq.)')
ax2.plot(bac.index, bac.values, color=C['bac'], lw=0.9, label='BACKUSI1 (der.)')
ax.set_ylabel('ALICORC1 (S/.)', color=C['ali'])
ax2.set_ylabel('BACKUSI1 (S/.)', color=C['bac'])
ax.set_title('Precios históricos — movimiento conjunto', fontsize=10)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=8)
ax.grid(axis='y', alpha=0.3)

ax3 = axes[1]
ax3.scatter(ali.values, bac.values, color=C['ali'], alpha=0.3, s=6)
# Línea de regresión de cointegración
X_reg = add_constant(ali.values)
res_ols = OLS(bac.values, X_reg).fit()
x_line = np.linspace(ali.min(), ali.max(), 100)
ax3.plot(x_line, res_ols.params[0] + res_ols.params[1]*x_line,
         color=C['spread'], lw=1.5, label=f'β={res_ols.params[1]:.3f}')
ax3.set_xlabel('ALICORC1 (S/.)')
ax3.set_ylabel('BACKUSI1 (S/.)')
ax3.set_title(f'Scatter + regresión cointegración\nR²={res_ols.rsquared:.4f}', fontsize=10)
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('data/finance_eda.png', dpi=130, bbox_inches='tight')
plt.show()
print('✓ data/finance_eda.png')

In [ ]:
# ── CONSTRUIR EL SPREAD ───────────────────────────────────────────────────────

# Paso 1: regresión de cointegración (OLS)
X_reg = add_constant(ali.values)
res   = OLS(bac.values, X_reg).fit()
alpha_hat = res.params[0]
beta_hat  = res.params[1]

print(f'── Relación de cointegración ────────────────────────────────────')
print(f'  BACKUSI1 = {alpha_hat:.4f} + {beta_hat:.4f} × ALICORC1')
print(f'  R²       = {res.rsquared:.4f}')

# Paso 2: residuos = spread
spread_raw = (bac.values - (alpha_hat + beta_hat * ali.values)).flatten()
spread     = pd.Series(spread_raw, index=ali.index, name='Spread')

# Paso 3: ADF sobre el spread
adf_res = adfuller(spread, autolag='AIC')
print(f'\n── ADF sobre el spread ────────────────────────────────────────')
print(f'  ADF stat : {adf_res[0]:.4f}')
print(f'  p-valor  : {adf_res[1]:.6f}')
print(f'  → {"Spread estacionario ✓ — cointegración válida" if adf_res[1] < 0.05 else "Spread no estacionario"}')

# Paso 4: normalizar spread → z-score
# Usar ventana rodante para capturar cambios de régimen
roll_mean = spread.rolling(60).mean()
roll_std  = spread.rolling(60).std()
z_score   = (spread - roll_mean) / roll_std

print(f'\n── Spread normalizado (z-score rodante 60d) ────────────────────')
print(f'  Media z  : {z_score.mean():.4f}')
print(f'  Std z    : {z_score.std():.4f}')
print(f'  |z| > 2  : {(z_score.abs() > 2).mean():.1%} de los días (señal activa)')

In [ ]:
# ── SEÑALES DE TRADING ────────────────────────────────────────────────────────

# Umbrales
ENTRY = 2.0   # entrar cuando |z| > 2
EXIT  = 0.5   # cerrar cuando |z| < 0.5

# Generar señales
signal = pd.Series(0, index=z_score.index, name='signal')
position = 0

for i in range(len(z_score)):
    z = z_score.iloc[i]
    if np.isnan(z):
        signal.iloc[i] = 0
        continue
    if position == 0:  # sin posición
        if z < -ENTRY:
            position = 1   # Long spread (comprar Y, vender X)
        elif z > ENTRY:
            position = -1  # Short spread (vender Y, comprar X)
    elif position == 1:
        if z > -EXIT:
            position = 0  # cerrar long
    elif position == -1:
        if z < EXIT:
            position = 0  # cerrar short
    signal.iloc[i] = position

# P&L de la estrategia
spread_ret  = spread.diff()
strat_ret   = signal.shift(1) * spread_ret
cumulative  = strat_ret.cumsum()

n_trades = (signal.diff().abs() > 0).sum()
pnl_total = cumulative.iloc[-1]
sharpe    = strat_ret.mean() / strat_ret.std() * np.sqrt(252) if strat_ret.std() > 0 else 0

print(f'── Backtesting señales de trading ──────────────────────────────')
print(f'  Número de trades  : {n_trades}')
print(f'  P&L acumulado     : {pnl_total:.2f} S/.')
print(f'  Sharpe ratio      : {sharpe:.3f}')
print(f'  % días con posición: {(signal != 0).mean():.1%}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates

# ── DASHBOARD MEJORADO ────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 12))
fig.suptitle(
    'Cointegración + Pairs Trading — ALICORC1 vs. BACKUSI1 (BVL)\n'
    f'{SOURCE}',
    fontsize=12, fontweight='bold', y=0.94
)
# Aumentamos ligeramente el hspace para que respiren los paneles
gs = gridspec.GridSpec(4, 1, hspace=0.12, height_ratios=[2, 1.5, 1, 1])

# P1 — Precios normalizados
ax1 = fig.add_subplot(gs[0])
ali_norm = ali / ali.iloc[0] * 100
bac_norm = bac / bac.iloc[0] * 100
ax1.plot(ali.index, ali_norm, color=C['ali'], lw=1.2, label='ALICORC1 (base 100)')
ax1.plot(bac.index, bac_norm, color=C['bac'], lw=1.2, label='BACKUSI1 (base 100)')
ax1.set_ylabel('Precio (base 100)')
ax1.set_title('Panel 1 — Precios normalizados (base 100)', loc='left', fontsize=10, pad=4)
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)
# Al no poner set_xticklabels([]), Matplotlib lo oculta automáticamente por el sharex

# P2 — Spread + z-score
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.fill_between(z_score.index, z_score, 0,
                 where=z_score >= 0, alpha=0.35, color=C['fill_s'])
ax2.fill_between(z_score.index, z_score, 0,
                 where=z_score <  0, alpha=0.35, color=C['fill_l'])
ax2.plot(z_score.index, z_score, color=C['spread'], lw=1.0)
ax2.axhline( ENTRY, color=C['signal'], lw=1.2, ls='--', label=f'+{ENTRY} (Short spread)')
ax2.axhline(-ENTRY, color=C['signal'], lw=1.2, ls='--', label=f'-{ENTRY} (Long spread)')
ax2.axhline(0, color='black', lw=0.8)
ax2.set_ylabel('Z-score (60d)')
ax2.set_title('Panel 2 — Z-score del spread (ventana rodante 60d)', loc='left', fontsize=10, pad=4)
ax2.legend(fontsize=8, loc='upper right')
ax2.grid(axis='y', alpha=0.3)

# P3 — Señal de posición (Colores nítidos y step='post')
ax3 = fig.add_subplot(gs[2], sharex=ax1)
ax3.fill_between(signal.index, signal, 0,
                 where=signal > 0, alpha=0.85, color='#00a65a', # Verde sólido
                 step='post', label='Long spread')
ax3.fill_between(signal.index, signal, 0,
                 where=signal < 0, alpha=0.85, color='#dd4b39', # Rojo sólido
                 step='post', label='Short spread')
ax3.axhline(0, color='black', lw=0.8)
ax3.set_ylabel('Posición')
ax3.set_yticks([-1, 0, 1])
ax3.set_yticklabels(['Short', 'Flat', 'Long'], fontsize=9, fontweight='bold')
ax3.set_title('Panel 3 — Posición activa', loc='left', fontsize=10, pad=4)
ax3.legend(fontsize=8, loc='upper left')
ax3.grid(axis='y', alpha=0.3)

# P4 — P&L acumulado (Relleno condicional según ganancia/pérdida)
ax4 = fig.add_subplot(gs[3], sharex=ax1)
ax4.fill_between(cumulative.index, cumulative, 0,
                 where=cumulative >= 0, alpha=0.4, color='#00a65a', interpolate=True)
ax4.fill_between(cumulative.index, cumulative, 0,
                 where=cumulative <  0, alpha=0.4, color='#dd4b39', interpolate=True)
ax4.plot(cumulative.index, cumulative, color='#2c3e50', lw=1.2)
ax4.axhline(0, color='black', lw=0.8)
ax4.set_ylabel('P&L acum. (S/.)')
ax4.set_xlabel('Fecha')
ax4.set_title(f'Panel 4 — P&L acumulado (Sharpe={sharpe:.2f})', loc='left', fontsize=10, pad=4)
ax4.grid(axis='y', alpha=0.3)

# Formato de fechas en el eje X
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax4.xaxis.set_major_locator(mdates.AutoDateLocator())
ax4.tick_params(axis='x', rotation=30) # Rota las fechas para que no se pisen

# Usamos subplots_adjust en lugar de tight_layout para mejor control con suptitle
plt.subplots_adjust(top=0.90, bottom=0.1, left=0.08, right=0.95)
plt.savefig('data/finance_dashboard.png', dpi=140, bbox_inches='tight')
plt.show()
print('✓ data/finance_dashboard.png')

In [ ]:


# ── EXPORTAR ─────────────────────────────────────────────────────────────────
# Transformación final: Aseguramos que todas las variables sean estrictamente 1D
df_export = pd.DataFrame({
    'ali': np.squeeze(ali),
    'bac': np.squeeze(bac),
    'spread': np.squeeze(spread),
    'z_score': np.squeeze(z_score),
    'signal': np.squeeze(signal),
    'pnl_cum': np.squeeze(cumulative)
})

df_export.to_csv('data/finance_pairs_output.csv')

print('✓ data/finance_pairs_output.csv')
print('✓ data/finance_eda.png')
print('✓ data/finance_dashboard.png')

## Conclusiones — contexto financiero

| Concepto | En Pairs Trading BVL | Aplicación Supply Chain |
|----------|---------------------|-------------------------|
| **Cointegración** | Precios se mueven juntos largo plazo | Demandas de SKUs sustitutos cointegradas |
| **Spread = residuo** | BACKUSI1 - β·ALICORC1 | Demanda A - β·Demanda B |
| **Z-score > 2** | Short spread → vender caro, comprar barato | Redirigir stock del SKU sobre-demandado |
| **Velocidad ajuste α** | Qué tan rápido revierte el spread | Semanas para que el desequilibrio se corrija |
| **VECM** | Modelo dinámico del ajuste | Forecast conjunto de dos SKUs relacionados |

**Próximo:** `2_Supply_Adaptation.ipynb` — cointegración entre precios mayoristas de commodities peruanos (Minagri) y su aplicación a la gestión de inventario de categorías relacionadas.